# Internal Language Model Analysis

This notebook computes the ILM-based statistics reported in Section 7 of the paper.
It measures how strongly the decoder's entrenched masculine bias aligns with the full
model's predictions, and how much the ILM preference correlates with the model's
preference for the same gendered form.

## Inputs

| Name | Description | How to obtain |
|---|---|---|
| `summary_dataframe.tsv` | One row per gender term instance, with model predictions, ILM probabilities, flip thresholds, and training data term frequencies. | Output of `prepare_df_for_analysis_clean.ipynb` (`output_path`). |

## Output

Printed statistics corresponding to Section 7 and the appendix (Tables 8–10):
- Mean ± std of ILM masculine preference, overall and by predicted gender
- Count table: predicted gender × whether the ILM favours the predicted form (Table 2)
- Pearson correlations between model and ILM preference, overall and split by flip / predicted gender

## Imports

In [1]:
import csv

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

## Configuration

Set all paths and parameters in this cell before running the notebook.

In [3]:
LANG = "it"            # one of: es, fr, it
MODEL = "transformer"    # one of: transformer, conformer

# summary_dataframe.tsv – output of prepare_df_for_analysis_clean.ipynb (output_path)
SUMMARY_TSV = f"/path/to/results/{MODEL}/en-{LANG}/summary_dataframe.tsv"

## 1. Load and Filter Data

Load the summary dataframe, restrict to category-1 speaker-referring terms excluding
articles and prepositions (following Savoldi et al. 2022), and compute all derived columns
used in the analyses below.

In [10]:
df = pd.read_csv(SUMMARY_TSV, sep='\t', escapechar='\\', quoting=csv.QUOTE_NONE)

# Category 1 = speaker-referring terms; Art/Prep excluded following Savoldi et al. (2022)
filtered_df = df[(df.cat_nb == 1) & (df.pos != 'Art/Prep')].copy()

# True if occluding the top salient features causes a gender flip within the 0–20% range
filtered_df['flip'] = ~np.isnan(filtered_df['flip_percent'])

# True if the ILM assigns higher probability to the predicted form than to the swapped form
filtered_df['ilm_prefers_predicted'] = filtered_df['ilm_prob_predicted'] > filtered_df['ilm_prob_swap']

# Relative ILM probability of the predicted form vs. its alternative
filtered_df['ilm_pref_for_predicted'] = (
    filtered_df['ilm_prob_predicted']
    / (filtered_df['ilm_prob_predicted'] + filtered_df['ilm_prob_swap'])
)

# Relative full-model probability of the predicted form vs. its alternative
filtered_df['model_pref_for_predicted'] = (
    filtered_df['orig_prob_predicted']
    / (filtered_df['orig_prob_predicted'] + filtered_df['orig_prob_swap'])
)

# Predicted gender: reference gender when correct, opposite when incorrect
filtered_df['predicted_gender'] = filtered_df.apply(
    lambda x: x.cat_gender if x.correct else ('F' if x.cat_gender == 'M' else 'M'),
    axis=1,
)

# ILM masculine preference: flip predicted-form preference for feminine predictions
# so the value always expresses the ILM's preference for the masculine form
filtered_df['ilm_pref_for_masc'] = filtered_df.apply(
    lambda x: x['ilm_pref_for_predicted'] if x['predicted_gender'] == 'M'
              else 1 - x['ilm_pref_for_predicted'],
    axis=1,
)

# Full-model masculine preference derived from the pre-computed fem/masc probabilities
filtered_df['model_pref_for_masc'] = (
    filtered_df['orig_prob_masc']
    / (filtered_df['orig_prob_masc'] + filtered_df['orig_prob_fem'])
)

print(f"{len(filtered_df)} examples after filtering")

357 examples after filtering


## 2. ILM Masculine Preference

Mean ± std of the ILM's preference for masculine, overall and broken down by predicted
gender. Corresponds to Tables 8–10 in the appendix.

In [11]:
# All predictions
filtered_df['ilm_pref_for_masc'].mean(), filtered_df['ilm_pref_for_masc'].std()

(0.7365468845528511, 0.29642086877397195)

In [12]:
# Masculine predictions only
filtered_df[filtered_df.predicted_gender == 'M']['ilm_pref_for_masc'].mean(), \
filtered_df[filtered_df.predicted_gender == 'M']['ilm_pref_for_masc'].std()

(0.8549999136262136, 0.1800295878617973)

In [13]:
# Feminine predictions only
filtered_df[filtered_df.predicted_gender == 'F']['ilm_pref_for_masc'].mean(), \
filtered_df[filtered_df.predicted_gender == 'F']['ilm_pref_for_masc'].std()

(0.5767911545525928, 0.3444217486904695)

## 3. Predicted Gender vs. ILM Preference Direction

Count of examples by predicted gender and whether the ILM assigns higher probability to
the predicted form than to the alternative. Corresponds to Table 2.

In [14]:
filtered_df.groupby(['predicted_gender', 'ilm_prefers_predicted'])['id'].count() \
    .unstack() \
    .reindex([True, False], axis=1) \
    .rename(columns={True: 'Higher Prob.', False: 'Lower Prob.'})

ilm_prefers_predicted,Higher Prob.,Lower Prob.
predicted_gender,,
F,46,106
M,195,10


## 4. Correlation Between Model and ILM Preferences

Pearson correlations measuring how strongly the full model's preference for masculine tracks that of the ILM. A high correlation indicates the model relies heavily on its entrenched biases; a low correlation indicates the audio input plays a larger role.

In [16]:
# Correlation of model vs. ILM masculine preference (reported in Section 7)
filtered_df['model_pref_for_masc'].corr(filtered_df['ilm_pref_for_masc'], method='pearson')

0.46957077115716483